
<style>
.ormedian-callout {
  border-left: 6px solid #24ABCF;
  background: #F6F8FA;
  padding: 14px 18px;
  margin: 12px 0;
  border-radius: 4px;
  color: #0B1F3B;
}
.ormedian-warning {
  border-left: 6px solid #1281B2;
  background: #EEF5F8;
  padding: 14px 18px;
  margin: 12px 0;
  border-radius: 4px;
  color: #0B1F3B;
}
.ormedian-checkpoint {
  border: 1px solid #B9DCE8;
  background: #FFFFFF;
  padding: 14px 18px;
  margin: 12px 0;
  border-radius: 8px;
  color: #0B1F3B;
}
</style>

<p align="center">
  <img src="../assets/ormedian_session1_banner.png" alt="Ormedian AI Engineering Fundamentals Session 1" width="100%" />
</p>

# Live coding learner notebook

Complete this notebook during the Sunday session. Type the code, predict outputs before running cells and explain results in your own words.

<div class="ormedian-warning">
Using ChatGPT or another coding assistant is allowed only when you can explain the resulting code. Code you cannot explain is not completed work.
</div>



## Learning outcomes

By the end, I should be able to explain:

- The AI/ML/deep-learning relationship.
- Supervised classification.
- Example, feature, label and prediction.
- Train, validation and test data.
- Baseline, TF-IDF and logistic regression.
- Accuracy, precision, recall, F1 and confusion matrix.
- Overfitting, underfitting, leakage and error analysis.



# 0. Setup

Run this cell first. It imports libraries and finds the repository root.


In [16]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    raise RuntimeError("Start Jupyter from the repository folder.")

sys.path.insert(0, str(ROOT))

from src.data import clean_dataset, dataset_summary, get_splits, load_dataset, validate_dataset
from src.evaluation import classification_metrics, error_frame
from src.modelling import build_majority_baseline, build_text_model

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
print("Repository root:", ROOT)

Repository root: c:\Users\tunmi\Desktop\AI-learning-folder\AI-Engineering-Fundamentals-Student\Session-1



# 1. Frame the task

Complete these before training anything.

- **Input: A custumer support message** 
- **Output: One intent label**
- **User of the prediction: The system the message from the customer is routing to**
- **Action supported: Then the message is directed to the correct agent**
- **One costly mistake: negation or subtle words**
- **Initial success metric: Accuracy**

### Checkpoint

Why is this classification? Why is it supervised?

It is classification because it is in categories and not in numbers
It is supervised because every training example has the correct label.



# 2. Inspect raw data

A model should not be the first import we reach for. We first inspect the examples and labels.


In [17]:
RAW_DATA_PATH = ROOT / "data" / "raw" / "support_intents_raw.csv"
raw = load_dataset(RAW_DATA_PATH)

# TODO: print the number of rows, column names and first eight examples.
print("Number of rows:", len(raw))
print("Column names:", list(raw.columns))
print("First eight examples:")
print(raw.head(8))

Number of rows: 185
Column names: ['example_id', 'text', 'label', 'split', 'difficulty', 'source']
First eight examples:
  example_id                                               text  \
0    ref-027           Could you undo the charge on my account?   
1    inv-025                 What remains unpaid on my account?   
2    acc-010              How do I edit my profile information?   
3    acc-023              Please edit the organisation details.   
4    tec-002                The app crashes whenever I open it.   
5    tec-011                My verification code never arrives.   
6    ref-026    That card payment should not have gone through.   
7    inv-017  I need a copy of the invoice and its current s...   

               label       split difficulty             source  
0     refund_request        test       hard  curated_synthetic  
1     invoice_status  validation       hard  curated_synthetic  
2     account_update       train       easy  curated_synthetic  
3     account_u

In [97]:
# TODO: calculate:
# 1. number of blank text values after stripping whitespace
number_of_blank_text = (raw['text'].str.strip() == "").sum()
print("Number of blank text values after stripping whitespace:", number_of_blank_text)
# 2. exact duplicate text-label pairs
duplicate_text_label = raw.duplicated(subset=['text', 'label']).sum()
print("Number of exact duplicate text-label pairs:", duplicate_text_label)
# 3. unique labels as stored
unique_labels = raw['label'].unique()
print("Unique labels as stored:", unique_labels)
# 4. split counts
split_counts = raw['split'].value_counts()
print("Split counts:")

Number of blank text values after stripping whitespace: 1
Number of exact duplicate text-label pairs: 2
Unique labels as stored: <StringArray>
[   'refund_request',    'invoice_status',    'account_update',
 'technical_support',      'cancel_order',   'general_enquiry',
     'mystery_label',  ' account_update ']
Length: 8, dtype: str
Split counts:



Write the problems you found:

1. There are duplicate rows
2. After i ran the unique label, the duplicated words changed to another word
3. There was one whitespace and two duplicate text-label

Why can a more powerful model not repair invalid labels or leaked test examples?
A model only learns from the data you give it and If the labels are wrong or if test examples have already been seen during training, the model will learn the mistakes

In [98]:
# Use the supplied helper, then inspect the report.
cleaned_raw, cleaning_report = clean_dataset(raw)
cleaning_report

{'rows_before': 185,
 'blank_text_removed': 1,
 'invalid_labels_removed': 1,
 'duplicates_removed': 3,
 'rows_after': 180}

In [99]:
# TODO: validate cleaned_raw and print a dataset summary.



# 3. Load the processed dataset and identify vocabulary

For one row, identify:

- Example: One row of the dataset
- Feature: The text column
- Label/target: the label column
- Prediction (not available until after training): This is what the model will produce until after training
- Metadata: The columns


In [100]:
PROCESSED_DATA_PATH = ROOT / "data" / "processed" / "support_intents.csv"
data = load_dataset(PROCESSED_DATA_PATH)
validate_dataset(data)

# TODO: display one row and write which field is the feature and which is the label.


In [101]:
# TODO: calculate and display class counts.
# Optional: draw a bar chart with pandas and matplotlib.



### Leakage checkpoint

Why must `label` and `split` not be included as model inputs? Would `difficulty` definitely be available for a real new support message?



# 4. Split roles

- Training data is used to: Teach the model
- Validation data is used to: Try different ideas and choose the best
- Test data is used to: test the final output of the model to see how well it performs
- The test set should not guide daily changes because: it will memorise the training data which can lead to overfitting


In [ ]:
train, validation, test = get_splits(data)

# TODO: display the size of each split.
print("Train size:", len(train))
print("Validation size:", len(validation))
print("Test size:", len(test))
# TODO: assert that example IDs do not overlap across splits.
# train_ids = set(train['id'])
# validation_ids = set(validation['id'])
# test_ids = set(test['id'])

Train size: 120
Validation size: 30
Test size: 30



# 5. Majority baseline

Before running the code, predict what a majority classifier does and why its result is useful.



In [111]:
baseline = build_majority_baseline()

# TODO:
# 1. fit the baseline using the training labels
# 2. predict validation labels
# 3. calculate classification_metrics
# 4. display accuracy and macro F1



Interpretation:

- What did the baseline actually learn?
- What must the real model demonstrate?



# 6. TF-IDF plus logistic regression

TF-IDF converts text to sparse numerical features. Logistic regression learns weights connecting those features to intent labels.


In [118]:
unigram_model = build_text_model(ngram_range=(1, 1))

# TODO: fit on train["text"] and train["label"].
# TODO: predict validation labels.
# TODO: calculate and display validation metrics.
unigram_model = build_text_model(ngram_range=(1, 1))
unigram_model.fit(train["text"], train["label"])
unigram_pred = unigram_model.predict(validation["text"])
unigram_metrics = classification_metrics(validation["label"], unigram_pred)

print("Accuracy:", round(unigram_metrics["accuracy"], 4))
print("Macro F1:", round(unigram_metrics["macro_f1"], 4))

Accuracy: 0.7667
Macro F1: 0.7626


In [124]:
# TODO: create a pandas DataFrame from classification_report(..., output_dict=True).
# Use zero_division=0 and transpose the result with .T.
report_df = pd.DataFrame(
    classification_report(validation["label"], unigram_pred, output_dict=True, zero_division=0)
).T
display(report_df)

,precision,recall,f1-score,support
account_update,0.750000,0.600000,0.666667,5.000000
cancel_order,1.000000,0.600000,0.750000,5.000000
general_enquiry,0.571429,0.800000,0.666667,5.000000
invoice_status,1.000000,0.600000,0.750000,5.000000
refund_request,0.714286,1.000000,0.833333,5.000000
technical_support,0.833333,1.000000,0.909091,5.000000
accuracy,0.766667,0.766667,0.766667,0.766667
macro avg,0.811508,0.766667,0.762626,30.000000
weighted avg,0.811508,0.766667,0.762626,30.000000



Choose one class and explain its precision and recall in full sentences.


In [125]:
# TODO: draw a validation confusion matrix.
# Hints:
# labels = sorted(data["label"].unique())
# fig, ax = plt.subplots(figsize=(10, 8))
# ConfusionMatrixDisplay.from_predictions(...)



Interpret one off-diagonal cell:

> There were ___ messages whose true label was ___ but the model predicted ___.



# 7. Error analysis

Do not stop at the score. Inspect actual wrong predictions.


In [126]:
# TODO: use error_frame(texts, true_labels, predictions).
# Display every validation error.



Analyse at least three errors:

| Text | True label | Predicted label | Likely reason | Suggested improvement |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |



# 8. Controlled experiment

**Question:** Will adding bigrams improve validation macro F1?

**Hypothesis:**

**One variable changed:** `ngram_range` from `(1, 1)` to `(1, 2)`.


In [127]:
bigram_model = build_text_model(ngram_range=(1, 2))

# TODO: fit, predict and evaluate on validation.
# TODO: create a comparison table for baseline, unigram and bigram models.



## Experiment conclusion

- Unigram validation macro F1:
- Bigram validation macro F1:
- Change:
- Did the result support the hypothesis?
- What is one reasonable explanation?



# 9. Final test gate

Select the model using validation evidence. Then evaluate that selected configuration once on the test set.


In [128]:
# TODO:
# 1. select unigram or bigram using validation macro F1
# 2. predict test labels
# 3. report test accuracy and macro F1
# 4. display the test confusion matrix
# 5. display test errors



# 10. End-of-session explanation

Without copying definitions, answer:

1. What did the model use as input?
 model used text as input
2. What was the target?
The correct label
3. Why did we keep validation and test separate?
 we keep them seperate because validation is what we use to judge how well the model has performed during training while test the final output of data  
4. Why was the majority baseline useful?
it is use to show the minimum score we must beat
5. What did TF-IDF do?
it converts text into number features
6. Which metric helped most and why?
Macro F1 because it treats every class equally
7. What did one wrong prediction teach you?
one wrong prediction can mess up the whole model by cursing a confusion.
8. Give one example of leakage.
putting the same message in both train and test
9. What changed in the experiment?
10. What would you try next?
Collect more real customer messages
